<a href="https://colab.research.google.com/github/ciril7/Elkanio-Internship/blob/main/Excercise%202/House_Price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

from sklearn.metrics import mean_absolute_error, mean_squared_error

from google.colab import files

uploaded = files.upload()

# Load Dataset
df = pd.read_csv('/content/house_sales_corrupted.csv')

print("First 5 Rows:")
print(df.head())

print("\nMissing Values:")
print(df.isnull().sum())

# ---------------------------
# Handle Missing SalePrice
# ---------------------------

df['SalePrice'] = df['SalePrice'].fillna(df['SalePrice'].median())

# ---------------------------
# Correct Invalid Values
# ---------------------------

df['Bedrooms'] = df['Bedrooms'].apply(
    lambda x: df['Bedrooms'].median()
    if pd.isnull(x) or x < 1 or x > 10 else x
)

df['SquareFootage'] = df['SquareFootage'].apply(
    lambda x: df['SquareFootage'].median()
    if pd.isnull(x) or x <= 0 else x
)

df['AgeOfHouse'] = df['AgeOfHouse'].apply(
    lambda x: df['AgeOfHouse'].median()
    if pd.isnull(x) or x < 0 else x
)

# ---------------------------
# One Hot Encoding
# ---------------------------

df = pd.get_dummies(df,
                    columns=['City'],
                    drop_first=True)

# ---------------------------
# Features and Target
# ---------------------------

X = df.drop('SalePrice', axis=1)
y = df['SalePrice']

# Fill Remaining Missing Values
imputer = SimpleImputer(strategy='median')
X = imputer.fit_transform(X)

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Scaling for SVM
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---------------------------
# Models
# ---------------------------

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ),
    "SVM": SVR(),
    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    )
}

results = []

print("\nMODEL PERFORMANCE\n")

for name, model in models.items():

    if name == "SVM":
        model.fit(X_train_scaled, y_train)
        pred = model.predict(X_test_scaled)

    else:
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, pred)

    rmse = np.sqrt(mean_squared_error(y_test, pred))

    results.append([name, mae, rmse])

    print(name)
    print("MAE :", round(mae,2))
    print("RMSE:", round(rmse,2))
    print("-"*40)

# ---------------------------
# Best Model
# ---------------------------

results_df = pd.DataFrame(
    results,
    columns=['Model','MAE','RMSE']
)

print("\nComparison Table:")
print(results_df)

best_model = results_df.loc[
    results_df['RMSE'].idxmin()
]

print("\nBEST MODEL")
print(best_model)

# ---------------------------
# Report Summary
# ---------------------------

print("\nREPORT SUMMARY")
print("1. Missing SalePrice values handled using median.")
print("2. Invalid values corrected.")
print("3. City encoded using One-Hot Encoding.")
print("4. Five supervised models evaluated.")
print("5. Best model selected using lowest RMSE.")

Saving house_sales_corrupted.csv to house_sales_corrupted (1).csv
First 5 Rows:
   Bedrooms  Bathrooms  SquareFootage  AgeOfHouse   City  SalePrice
0         4          2           3060          38  CityB   604752.0
1         5          1           1819          55  CityC   498569.0
2         3          2           3465          62  CityB   576648.0
3         5          1           2355          45  CityB   548895.0
4         5          3           1872          87  CityD   567621.0

Missing Values:
Bedrooms          0
Bathrooms         0
SquareFootage     0
AgeOfHouse        0
City              0
SalePrice        50
dtype: int64

MODEL PERFORMANCE

Linear Regression
MAE : 25524.85
RMSE: 50527.13
----------------------------------------
Decision Tree
MAE : 44802.36
RMSE: 69469.53
----------------------------------------
Random Forest
MAE : 33600.03
RMSE: 55548.27
----------------------------------------
SVM
MAE : 87268.73
RMSE: 111151.16
----------------------------------------
Gradien